# CodexProbe on Kaggle

Records every LLM call the Codex CLI makes, first against OpenAI, then against a local Qwen2.5-Coder model on Kaggle's GPU, and compares the two traces (cost, latency, agent behaviour, task success).

**Before running:**
1. Settings → Accelerator: **GPU T4 x2**; Internet: **On** (needs a phone-verified Kaggle account).
2. Add-ons → Secrets → add `OPENAI_API_KEY` and attach it to this notebook. Without it, run (a) is skipped and only the Qwen run happens. The key never appears in the logs (only its last 4 characters).


## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Install CodexProbe and the Codex CLI

In [ ]:
import os, shutil, subprocess
REPO = "https://github.com/Rahul-Verma97/codex-probe.git"
if os.path.isdir("/kaggle/input/codex-probe/codex_probe"):   # optional: a Kaggle Dataset copy
    !cp -r /kaggle/input/codex-probe/codex_probe /kaggle/working/codex_probe
elif not os.path.isdir("/kaggle/working/codex_probe"):
    !git clone -q {REPO} /kaggle/working/codex_probe
!pip install -q -e /kaggle/working/codex_probe
# Codex CLI is a Node package; install Node 22 if it's missing.
if shutil.which("npm") is None:
    !curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null && apt-get install -y -qq nodejs > /dev/null
!npm i -g @openai/codex > /dev/null 2>&1
!codex --version


## 3. Start Ollama and pull Qwen2.5-Coder-7B
Codex only speaks the **Responses API** now (`wire_api = "chat"` is rejected), so the local server must expose `/v1/responses`. Recent Ollama releases do.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import os, time
ollama = subprocess.Popen(["ollama", "serve"], stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT,
                          env=dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_CONTEXT_LENGTH="32768"))
time.sleep(5)
!ollama --version
!ollama pull qwen2.5-coder:7b

In [ ]:
# Smoke test: the Responses endpoint must answer, or Codex can't use this backend.
!curl -s http://127.0.0.1:11434/v1/responses -H 'Content-Type: application/json' \
  -d '{"model":"qwen2.5-coder:7b","input":"Say hi in one word"}' | head -c 600

## 4. Run (a): Codex → CodexProbe → OpenAI

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["OPENAI_API_KEY"] = UserSecretsClient().get_secret("OPENAI_API_KEY")
except Exception as exc:
    print("OPENAI_API_KEY secret not available, skipping run (a):", exc)
%cd /kaggle/working/codex_probe
# Change --model to any Codex-capable model your key can use.
if os.environ.get("OPENAI_API_KEY"):
    !python examples/quickstart.py --config configs/openai.json --model gpt-5-codex --sandbox danger-full-access


## 5. Run (b): same recorder, backend swapped by config → local Qwen

In [ ]:
!python examples/quickstart.py --config configs/qwen_ollama.json --model qwen2.5-coder:7b \
    --env-key OLLAMA_DUMMY_KEY --sandbox danger-full-access --timeout 1200

## 6. Inspect the logs

In [ ]:
import glob, json
from codex_probe import load_session
for path in sorted(glob.glob("codex_probe_logs/*.jsonl")):
    calls = load_session(path)          # also validates every line against the schema
    print(f"\n== {path}: {len(calls)} calls")
    for c in calls:
        print(f"  #{c['call_index']} {c['status']} {c['upstream_url']} {c['latency_ms']:.0f}ms")
!codex-probe show $(ls codex_probe_logs/qwen-*.jsonl | head -1)

In [ ]:
# One full captured call: instructions, full input history, tool schema, and the reassembled response.
first = load_session(sorted(glob.glob("codex_probe_logs/qwen-*.jsonl"))[0])[0]
req = first["request_body"]
print("request keys:", list(req))
print("tools offered:", [t.get("name") or t.get("type") for t in req.get("tools", [])])
print("input roles:", [i.get("role") or i.get("type") for i in req.get("input", [])])
print(json.dumps(first["response_body"], indent=2)[:3000])

## 7. Compare the backends
Same task, same Codex version, same recorder; only the config changed. Metrics come from the logs alone (see `src/codex_probe/analysis.py`): tokens and prompt growth (Codex re-sends the full history every turn), share of each request that is re-sent history, latency and throughput, tool-call validity, repeated actions, and whether the independent check passed.


In [ ]:
import glob, json
latest = {}
for name in ("openai", "qwen"):
    logs = sorted(glob.glob(f"codex_probe_logs/{name}-*.jsonl"))
    if logs:
        latest[name] = logs[-1]
if latest:
    args = " ".join(latest.values())
    names = " ".join(latest)
    !codex-probe analyze {args} --names {names} --out comparison.md --json comparison.json
    for name, log in latest.items():
        outcome = log[:-len(".jsonl")] + ".outcome.json"
        if os.path.exists(outcome):
            o = json.load(open(outcome))
            print(f"{name}: solved={o['solved']} codex_exit={o['codex_exit_code']} files={o['files']}")
    from IPython.display import Markdown, display
    display(Markdown(open("comparison.md").read()))


In [ ]:
# Keep the evidence as notebook output files.
!mkdir -p /kaggle/working/results && cp -r codex_probe_logs comparison.md comparison.json /kaggle/working/results/ 2>/dev/null; ls -la /kaggle/working/results


## Optional: vLLM instead of Ollama
vLLM also serves `/v1/responses`. On T4 GPUs use fp16 and both cards. Newer vLLM releases may not support T4 (compute 7.5); if it fails to start, stay with Ollama and note it in the write-up.
```bash
pip install vllm
vllm serve Qwen/Qwen2.5-Coder-7B-Instruct --dtype half --tensor-parallel-size 2 --max-model-len 32768 \
     --enable-auto-tool-choice --tool-call-parser hermes --port 8000
python examples/quickstart.py --config configs/qwen_vllm.json --model Qwen/Qwen2.5-Coder-7B-Instruct --env-key VLLM_DUMMY_KEY --sandbox danger-full-access
```